# T39 - 成交数据
## 第三章：数据处理

本章介绍成交数据的处理流程，包括：
1. 数据加载
2. 数据清洗
3. 数据转换
4. 数据聚合

## 1. 导入必要的库

In [ ]:
# 数据处理
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import logging
from typing import Dict, List, Optional

# 数据库连接
import sqlalchemy
from sqlalchemy.sql import text

# 导入配置
from config import config

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger('数据处理')

print('导入成功！')

## 2. 数据加载模块

In [ ]:
class TradeDataLoader:
    """成交数据加载器"""
    
    def __init__(self):
        self.engine = None
    
    def _get_engine(self):
        """获取数据库引擎"""
        if self.engine is None:
            self.engine = sqlalchemy.create_engine(
                config.database.get_mysql_yq_connection_string(),
                poolclass=sqlalchemy.pool.NullPool
            )
        return self.engine
    
    def load_all_data(self) -> pd.DataFrame:
        """加载所有成交数据"""
        try:
            engine = self._get_engine()
            df = pd.read_sql('SELECT * FROM yq.cjqb', engine)
            logger.info(f'加载数据: {len(df)} 条')
            return df
        except Exception as e:
            logger.exception('加载数据失败')
            return pd.DataFrame()
    
    def load_data_by_date_range(self, start_date: str, end_date: str) -> pd.DataFrame:
        """
        加载指定日期范围的数据
        
        Args:
            start_date: 开始日期 (YYYY-MM-DD)
            end_date: 结束日期 (YYYY-MM-DD)
        """
        try:
            engine = self._get_engine()
            query = text('''
                SELECT * FROM yq.cjqb 
                WHERE tradedDate >= :start_date AND tradedDate <= :end_date
            ''')
            df = pd.read_sql(query, engine, params={'start_date': start_date, 'end_date': end_date})
            logger.info(f'加载 {start_date} ~ {end_date} 数据: {len(df)} 条')
            return df
        except Exception as e:
            logger.exception('加载数据失败')
            return pd.DataFrame()
    
    def load_recent_data(self, days: int = 30) -> pd.DataFrame:
        """
        加载最近N天的数据
        
        Args:
            days: 天数
        """
        end_date = datetime.now().strftime('%Y-%m-%d')
        start_date = (datetime.now() - timedelta(days=days)).strftime('%Y-%m-%d')
        return self.load_data_by_date_range(start_date, end_date)

# 创建加载器实例
loader = TradeDataLoader()
print('数据加载器初始化完成')

## 3. 数据清洗模块

In [ ]:
class TradeDataCleaner:
    """成交数据清洗器"""
    
    def __init__(self):
        self.price_min = config.trade_data.price_min
        self.price_max = config.trade_data.price_max
        self.amount_min = config.trade_data.amount_min
    
    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        清洗成交数据
        
        Args:
            df: 原始数据
            
        Returns:
            pd.DataFrame: 清洗后的数据
        """
        if df.empty:
            return df
        
        original_count = len(df)
        
        # 1. 复制数据
        df = df.copy()
        
        # 2. 统一日期格式
        if 'tradedDate' in df.columns:
            df['tradedDate'] = pd.to_datetime(df['tradedDate'])
            df['dt'] = df['tradedDate'].dt.strftime('%Y-%m-%d')
        
        # 3. 处理数值字段
        numeric_fields = ['tradedPrice', 'tradedAmount', 'tradedYield']
        for field in numeric_fields:
            if field in df.columns:
                df[field] = pd.to_numeric(df[field], errors='coerce')
        
        # 4. 删除价格为空或异常的记录
        if 'tradedPrice' in df.columns:
            df = df[
                (df['tradedPrice'].notna()) &
                (df['tradedPrice'] > self.price_min) &
                (df['tradedPrice'] < self.price_max)
            ]
        
        # 5. 删除成交金额为空或异常的记录
        if 'tradedAmount' in df.columns:
            df = df[
                (df['tradedAmount'].notna()) &
                (df['tradedAmount'] > self.amount_min)
            ]
        
        # 6. 删除重复记录
        df = df.drop_duplicates()
        
        # 7. 重置索引
        df = df.reset_index(drop=True)
        
        cleaned_count = len(df)
        logger.info(f'数据清洗完成: {original_count} -> {cleaned_count} (删除 {original_count - cleaned_count} 条)')
        
        return df

# 创建清洗器实例
cleaner = TradeDataCleaner()
print('数据清洗器初始化完成')

## 4. 数据转换模块

In [ ]:
class TradeDataTransformer:
    """成交数据转换器"""
    
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        转换成交数据格式
        
        Args:
            df: 清洗后的数据
            
        Returns:
            pd.DataFrame: 转换后的数据
        """
        if df.empty:
            return df
        
        df = df.copy()
        
        # 1. 添加成交金额分类
        if 'tradedAmount' in df.columns:
            thresholds = config.trade_data.volume_thresholds
            df['amount_category'] = pd.cut(
                df['tradedAmount'],
                bins=[0, thresholds['small'], thresholds['medium'], thresholds['large'], float('inf')],
                labels=['小', '中', '大', '超大']
            )
        
        # 2. 添加年份、月份、季度列
        if 'tradedDate' in df.columns:
            df['year'] = df['tradedDate'].dt.year
            df['month'] = df['tradedDate'].dt.month
            df['quarter'] = df['tradedDate'].dt.quarter
            df['week'] = df['tradedDate'].dt.isocalendar().week
            df['weekday'] = df['tradedDate'].dt.dayofweek  # 0=周一, 6=周日
        
        # 3. 计算收益率变化（如果有收益率数据）
        if 'tradedYield' in df.columns and 'bondCode' in df.columns:
            df = df.sort_values(['bondCode', 'tradedDate'])
            df['yield_change'] = df.groupby('bondCode')['tradedYield'].diff()
        
        logger.info(f'数据转换完成，新增列: amount_category, year, month, quarter, week, weekday, yield_change')
        
        return df

# 创建转换器实例
transformer = TradeDataTransformer()
print('数据转换器初始化完成')

## 5. 数据聚合模块

In [ ]:
class TradeDataAggregator:
    """成交数据聚合器"""
    
    def aggregate_by_date(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        按日期聚合数据
        
        Args:
            df: 转换后的数据
            
        Returns:
            pd.DataFrame: 按日期聚合的数据
        """
        if df.empty:
            return pd.DataFrame()
        
        agg_dict = {
            'tradedAmount': ['sum', 'mean', 'count'],
            'tradedPrice': ['mean', 'std', 'min', 'max'],
            'tradedYield': ['mean', 'std'],
            'bondCode': 'nunique'
        }
        
        # 只保留存在的列
        available_agg = {k: v for k, v in agg_dict.items() if k in df.columns}
        
        daily_agg = df.groupby('dt').agg(available_agg)
        daily_agg.columns = ['_'.join(col).strip() for col in daily_agg.columns.values]
        daily_agg = daily_agg.reset_index()
        
        # 重命名列
        rename_dict = {
            'tradedAmount_sum': 'total_amount',
            'tradedAmount_mean': 'avg_amount',
            'tradedAmount_count': 'trade_count',
            'tradedPrice_mean': 'avg_price',
            'tradedPrice_std': 'price_std',
            'tradedPrice_min': 'min_price',
            'tradedPrice_max': 'max_price',
            'tradedYield_mean': 'avg_yield',
            'tradedYield_std': 'yield_std',
            'bondCode_nunique': 'bond_count'
        }
        daily_agg = daily_agg.rename(columns=rename_dict)
        
        logger.info(f'按日期聚合完成: {len(daily_agg)} 天')
        return daily_agg
    
    def aggregate_by_bond(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        按债券聚合数据
        
        Args:
            df: 转换后的数据
            
        Returns:
            pd.DataFrame: 按债券聚合的数据
        """
        if df.empty or 'bondCode' not in df.columns:
            return pd.DataFrame()
        
        agg_dict = {
            'tradedAmount': ['sum', 'mean', 'count'],
            'tradedPrice': ['mean', 'std'],
            'tradedYield': ['mean', 'std'],
            'dt': ['min', 'max', 'nunique']
        }
        
        # 只保留存在的列
        available_agg = {k: v for k, v in agg_dict.items() if k in df.columns}
        
        bond_agg = df.groupby('bondCode').agg(available_agg)
        bond_agg.columns = ['_'.join(col).strip() for col in bond_agg.columns.values]
        bond_agg = bond_agg.reset_index()
        
        # 重命名列
        rename_dict = {
            'tradedAmount_sum': 'total_amount',
            'tradedAmount_mean': 'avg_amount',
            'tradedAmount_count': 'trade_count',
            'tradedPrice_mean': 'avg_price',
            'tradedPrice_std': 'price_std',
            'tradedYield_mean': 'avg_yield',
            'tradedYield_std': 'yield_std',
            'dt_min': 'first_trade_date',
            'dt_max': 'last_trade_date',
            'dt_nunique': 'trade_days'
        }
        bond_agg = bond_agg.rename(columns=rename_dict)
        
        logger.info(f'按债券聚合完成: {len(bond_agg)} 只债券')
        return bond_agg
    
    def aggregate_by_rating(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        按评级聚合数据
        
        Args:
            df: 转换后的数据
            
        Returns:
            pd.DataFrame: 按评级聚合的数据
        """
        if df.empty or 'yyRating' not in df.columns:
            return pd.DataFrame()
        
        agg_dict = {
            'tradedAmount': ['sum', 'mean', 'count'],
            'tradedYield': ['mean', 'std'],
            'bondCode': 'nunique'
        }
        
        # 只保留存在的列
        available_agg = {k: v for k, v in agg_dict.items() if k in df.columns}
        
        rating_agg = df.groupby('yyRating').agg(available_agg)
        rating_agg.columns = ['_'.join(col).strip() for col in rating_agg.columns.values]
        rating_agg = rating_agg.reset_index()
        
        logger.info(f'按评级聚合完成: {len(rating_agg)} 个评级')
        return rating_agg

# 创建聚合器实例
aggregator = TradeDataAggregator()
print('数据聚合器初始化完成')

## 6. 数据处理流水线

In [ ]:
class TradeDataPipeline:
    """成交数据处理流水线"""
    
    def __init__(self):
        self.loader = TradeDataLoader()
        self.cleaner = TradeDataCleaner()
        self.transformer = TradeDataTransformer()
        self.aggregator = TradeDataAggregator()
    
    def process(self, days: int = 30) -> Dict[str, pd.DataFrame]:
        """
        执行完整的数据处理流程
        
        Args:
            days: 处理最近N天的数据
            
        Returns:
            Dict[str, pd.DataFrame]: 处理结果
        """
        print(f'\n=== 开始处理最近 {days} 天的数据 ===')
        
        # 1. 加载数据
        print('\n1. 加载数据...')
        df = self.loader.load_recent_data(days)
        if df.empty:
            print('没有数据需要处理')
            return {}
        print(f'   加载 {len(df)} 条记录')
        
        # 2. 清洗数据
        print('\n2. 清洗数据...')
        df_clean = self.cleaner.clean(df)
        print(f'   清洗后 {len(df_clean)} 条记录')
        
        # 3. 转换数据
        print('\n3. 转换数据...')
        df_transformed = self.transformer.transform(df_clean)
        print(f'   转换后 {len(df_transformed)} 条记录')
        
        # 4. 聚合数据
        print('\n4. 聚合数据...')
        daily_agg = self.aggregator.aggregate_by_date(df_transformed)
        bond_agg = self.aggregator.aggregate_by_bond(df_transformed)
        rating_agg = self.aggregator.aggregate_by_rating(df_transformed)
        
        print('\n=== 处理完成 ===')
        
        return {
            'raw': df,
            'cleaned': df_clean,
            'transformed': df_transformed,
            'daily_agg': daily_agg,
            'bond_agg': bond_agg,
            'rating_agg': rating_agg
        }

# 创建流水线实例
pipeline = TradeDataPipeline()
print('处理流水线初始化完成')

## 7. 执行数据处理

In [ ]:
# 执行数据处理流水线
results = pipeline.process(days=30)

# 显示处理结果概览
if results:
    print('\n=== 处理结果概览 ===')
    print(f'原始数据: {len(results["raw"])} 条')
    print(f'清洗后: {len(results["cleaned"])} 条')
    print(f'转换后: {len(results["transformed"])} 条')
    print(f'按日期聚合: {len(results["daily_agg"])} 天')
    print(f'按债券聚合: {len(results["bond_agg"])} 只债券')
    print(f'按评级聚合: {len(results["rating_agg"])} 个评级')

## 8. 查看聚合结果

In [ ]:
# 查看按日期聚合的结果
if results and not results['daily_agg'].empty:
    print('\n=== 按日期聚合结果（前10条）===')
    print(results['daily_agg'].head(10).to_string())

In [ ]:
# 查看按债券聚合的结果（按成交金额排序）
if results and not results['bond_agg'].empty:
    print('\n=== 成交金额前10的债券 ===')
    top_bonds = results['bond_agg'].nlargest(10, 'total_amount')
    print(top_bonds.to_string())

In [ ]:
# 查看按评级聚合的结果
if results and not results['rating_agg'].empty:
    print('\n=== 按评级聚合结果 ===')
    print(results['rating_agg'].to_string())

## 9. 数据质量报告

In [ ]:
def generate_quality_report(results: Dict[str, pd.DataFrame]):
    """生成数据质量报告"""
    if not results or results['transformed'].empty:
        print('没有数据')
        return
    
    df = results['transformed']
    
    print('\n' + '='*60)
    print('数据质量报告')
    print('='*60)
    
    # 基本统计
    print('\n1. 基本统计')
    print(f'   总记录数: {len(df):,}')
    print(f'   日期范围: {df["dt"].min()} ~ {df["dt"].max()}')
    print(f'   债券数量: {df["bondCode"].nunique():,}')
    
    # 缺失值统计
    print('\n2. 缺失值统计')
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    for col in missing[missing > 0].index:
        print(f'   {col}: {missing[col]} ({missing_pct[col]}%)')
    
    if missing.sum() == 0:
        print('   无缺失值')
    
    # 数值字段统计
    print('\n3. 数值字段统计')
    numeric_cols = ['tradedAmount', 'tradedPrice', 'tradedYield']
    for col in numeric_cols:
        if col in df.columns:
            stats = df[col].describe()
            print(f'\n   {col}:')
            print(f'      均值: {stats["mean"]:,.2f}')
            print(f'      标准差: {stats["std"]:,.2f}')
            print(f'      最小值: {stats["min"]:,.2f}')
            print(f'      最大值: {stats["max"]:,.2f}')
    
    print('\n' + '='*60)

generate_quality_report(results)